[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/generative-llms.ipynb)

# Chapter 2: Generative LLMs for RAG

In a RAG pipeline, the generative LLM is the final step — it takes retrieved context and the user's query, then produces a natural language answer. The quality of this generation step depends heavily on the prompt template and the model's ability to stay grounded in the provided context.

This notebook demonstrates using Google's Gemini model (through LangChain's `ChatGoogleGenerativeAI`) to generate responses in a RAG pipeline with prompt templates.

**What you'll learn:**
- Structure a RAG prompt template with query and context placeholders
- Use the Gemini API to generate grounded responses
- Test how the same context answers different queries

**Prerequisites:** `pip install langchain-google-genai python-dotenv` and set `GOOGLE_API_KEY` in your environment or a `.env` file.

## Setup

We load API credentials and define a prompt template with placeholders for the query and retrieved context.

In [4]:
import os

import dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import SecretStr

# Load GOOGLE_API_KEY from a .env file if one exists; an already-set
# environment variable is not overwritten.
dotenv.load_dotenv()

# Read the key once here and pass it to the client. Without this, the client
# reads GOOGLE_API_KEY on its own and silently falls back to GEMINI_API_KEY.
if "GOOGLE_API_KEY" not in os.environ:
    raise RuntimeError("Set the GOOGLE_API_KEY environment variable before running this notebook.")
google_api_key: SecretStr = SecretStr(os.environ["GOOGLE_API_KEY"])

CHAT_MODEL: str = "gemini-3-flash-preview"

llm: ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    api_key=google_api_key,
    # Google recommends 1.0 for Gemini 3 models: lower values can cause
    # looping or degraded answers.
    temperature=1.0,
    thinking_level="low",
    # None means no timeout: a stalled request waits forever.
    timeout=None,
    max_retries=6,
)

In [5]:

query = "If I don't care about budget or time, what's the most accurate way to judge the RAG system?"

context = """
The next step is to build the critics for evaluating the responses from RAG. A common practice is called **LLM-as-a-judge**. 
Once again, we leverage the instruction following capability of LLMs to configure them into evaluators that judge different aspects of the responses. 

There are usually two common aspects: 
**relevance** -- does the response answer the query? and 
**faithfulness** -- is the response well supported by the retrieved documents? 

LLM-as-a-judge has several drawbacks. 
First, it maybe slow and expensive because it uses expensive LLMs. 
Second, it may not always be accurate. LLMs can hallucinate, e.g., the reasoning from one step to another may be flawed. 

An alternative to LLM-as-a-judge is dedicated **NLG (natural language generation) evaluation models** which are generally much smaller than an LLM. 
Finetuned to evaluate specific aspects of LLM generation, these models are much faster, cheaper, and more robust than LLM-as-a-judge. 
For example, Vectara's HHEM is a time-tested model to judge faithfulness. 
It has been downloaded more than 5 million times between August 2024 (when it debut) and August 2025. 

The last, but golden evaluation is **human evaluation**. 
Human evaluators can provide the most accurate and nuanced assessments of LLM responses, taking into account context, intent, and subtlety that automated systems may miss.
However, human evaluation is also the most resource-intensive and time-consuming method. 
It is often used as a final, small-scale check after automated evaluations have been performed.
"""

prompt_template= """
You are a good reader. Answer the query based on the context provided. Give me a short answer. 

Query: {query}
Context: {context}

"""

## Query the LLM

We send the filled prompt to Gemini and test how the same context answers different questions, showing the generation step of a RAG pipeline.

In [6]:
def answer_query(query: str, context: str) -> str:
    """Fill the RAG prompt template, send it to Gemini, and return the reply text."""
    filled_prompt: str = prompt_template.format(query=query, context=context)
    message: HumanMessage = HumanMessage(content=filled_prompt)
    response: AIMessage = llm.invoke([message])

    # Gemini 3 replies can hold several content blocks (for example, thinking
    # signatures). .text joins only the text blocks into one string.
    reply_text: str = response.text
    return reply_text


print(answer_query(query, context))

The most accurate way to judge the RAG system is **human evaluation**, as human evaluators provide the most nuanced assessments and capture subtleties that automated systems may miss.


In [7]:
# A different query
query = "What are the common aspects that people judge the RAG system on?"

print(answer_query(query, context))

Based on the context provided, the two common aspects used to judge a RAG system are:

1.  **Relevance:** Whether the response actually answers the query.
2.  **Faithfulness:** Whether the response is well-supported by the retrieved documents.


In [8]:
# Yet another different query
query = "How many ways to evaluate the RAG system? Just method names."

print(answer_query(query, context))

Based on the context, there are three ways:

1. **LLM-as-a-judge**
2. **NLG evaluation models**
3. **Human evaluation**
